In [1]:
# Core
import torch
import numpy as np

# Models
from models.hf_model import HFModel
from models.outputs import ModelOutput

# Uncertainty
from uncertainty.whitebox_uncertainty import WhiteBoxUncertainty
from uncertainty.graybox_uncertainty import GrayBoxUncertainty
from uncertainty.black_uncertainty import BlackBoxUncertainty

# Evaluation
from evaluation.aggregation import Aggregator
from evaluation.hallucination_score import HallucinationScore
from evaluation.thresholds import HallucinationThresholds
from evaluation.report import EvaluationReport

# Decision
from decision.final_score import FinalScore
from decision.hallucination_decider import HallucinationDecider


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = HFModel("gpt2")  # token yoksa open model

prompt = "What is the capital of France?"

output = model.generate(
    prompt,
    max_new_tokens=40,
    num_return_sequences=3,
)

output.responses


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 619.53it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


['What is the capital of France?\n\nNot that I believe it. Of course not. This may be the case for a number of reasons: the capital of France is the capital of three nationalities, of which the French capital',
 'What is the capital of France? To what extent is this state or a colony a sovereign? What is its relationship with the state? A country is considered as a whole. It is not a one-world. Everything should be united',
 'What is the capital of France? And does the royal family ever have a capital which has become unclaimed, or have they managed to gain it?"\n\n"Yes."\n\nShe looked up at the wall. "My name']

In [3]:
if output.has_whitebox():
    white = WhiteBoxUncertainty(
        scores=output.logits,
        token_ids=output.token_ids,
        text_responses=output.responses
    )

    white_entropy = white.predictive_entropy()
    white_conf = white.confidence()
    white_cons = white.self_consistency()

    print("White Entropy:", white_entropy)
    print("White Confidence:", white_conf)
    print("White Consistency:", white_cons)
else:
    white_entropy = None


White Entropy: 2.184349536895752
White Confidence: 3.0134031709054e-36
White Consistency: 0.3333333333333333


In [4]:
gray = GrayBoxUncertainty(
    responses=output.responses,
    log_probs=output.log_probs
)

gray_conf = gray.confidence()
gray_entropy = gray.response_entropy()

print("Gray Confidence:", gray_conf)
print("Gray Entropy:", gray_entropy)


Gray Confidence: 0.029431638255528976
Gray Entropy: 1.0986122886651097


In [5]:
black = BlackBoxUncertainty(output.responses)

black_conf = black.confidence()
black_entropy = black.response_entropy()
black_unique = black.unique_ratio()

print("Black Confidence:", black_conf)
print("Black Entropy:", black_entropy)
print("Black Unique Ratio:", black_unique)


Black Confidence: 0.3333333333333333
Black Entropy: 1.0986122886651097
Black Unique Ratio: 1.0


In [6]:
consistency = Aggregator.consistency_score(output.responses)
unique_ratio = Aggregator.unique_ratio(output.responses)

print("Aggregator Consistency:", consistency)
print("Aggregator Unique Ratio:", unique_ratio)


Aggregator Consistency: 0.3333333333333333
Aggregator Unique Ratio: 1.0


In [7]:
hs = HallucinationScore(
    white_score=white_entropy if output.has_whitebox() else None,
    gray_score=gray_conf,
    black_score=black_conf,
)

hallucination_score = hs.score()
hallucination_score


1.2738615217689633

In [8]:
risk_level = HallucinationThresholds.interpret(hallucination_score)
risk_level


'CRITICAL'

In [9]:
metrics = {
    "entropy": white_entropy if white_entropy is not None else 0,
    "self_consistency": black_conf,
    "confidence": gray_conf
}

final_score = FinalScore().compute(metrics)

decider = HallucinationDecider(
    thresholds={"hallucination": 0.6}
)

decision = decider.decide(
    {"final_score": final_score}
)

final_score, decision


(1.01295947574274, 'hallucination')

In [10]:
report = EvaluationReport(
    hallucination_score=final_score,
    level=risk_level,
    white=white_entropy,
    gray=gray_conf,
    black=black_conf
)

report.pretty_print()
report.to_dict()


--- Evaluation Report ---
Hallucination Score : 1.01295947574274
Risk Level : CRITICAL
White-box Entropy   : 2.1843
Gray-box Confidence : 0.0294
Black-box Consist.  : 0.3333


{'hallucination_score': 1.01295947574274,
 'risk_level': 'CRITICAL',
 'white_uncertainty': 2.184349536895752,
 'gray_uncertainty': 0.029431638255528976,
 'black_uncertainty': 0.3333333333333333}